# 01 · Quickstart — your first Mock Data Challenge

One BBH signal + Einstein Telescope noise, from a YAML config to frame
files you can open with GWpy. **Config in, data out** — this is the core
gwmock workflow; everything later refines one of its ingredients.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260706-leuven-gw-workshop-gwmock/blob/main/materials/notebooks/01-quickstart-mdc.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in the correlated-noise machinery used for the
# stochastic-background examples.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 1. The mental model

A gwmock config has two parts:

- **`globals`** — sampling frequency, duration, start time, seed, directories.
- **`orchestration`** — up to three stages:
    - `population`: where source parameters come from,
    - `signal`: waveform model + detectors + output files,
    - `noise`: noise model + output files.

First, the population: a plain CSV, one row per event.

In [ ]:
%%writefile population.csv
detector_frame_mass_1,detector_frame_mass_2,coa_time,distance,declination,right_ascension,polarization_angle,inclination
30.0,25.0,1577491300.0,400.0,0.3,1.1,0.2,0.5

A 30+25 M☉ BBH at 400 Mpc, coalescing 82 s into our segment.

Now the config: 128 s of 2048 Hz data for the **ET 2L aligned**
configuration. `{{ detectors }}`, `{{ start_time }}`, `{{ duration }}`
are template variables — gwmock expands the network alias into the two
interferometers and renders one file per detector.

In [ ]:
%%writefile config.yaml
globals:
    simulator-arguments:
        sampling-frequency: 2048
        duration: 128
        total-duration: 128
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output
    metadata-directory: metadata
orchestration:
    population:
        backend: FilePopulationLoader
        source-type: bbh
        arguments:
            path: population.csv
    signal:
        waveform-model: IMRPhenomXPHM
        minimum-frequency: 20
        earth-rotation: true
        detectors:
            - ET-2L-Aligned
        output:
            output_directory: signal
            file_name: 'E-{{ detectors }}_BBH-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'
    noise:
        arguments:
            psd_file: ET_10_full_cryo_psd
            seed: 42
        output:
            output_directory: noise
            file_name: 'E-{{ detectors }}_NOISE-{{ start_time }}-{{ duration }}.gwf'
            arguments:
                channel: '{{ detectors }}:STRAIN'


## 2. Validate the plan, then run

`--dry-run` checks the whole plan without writing data:

In [ ]:
!gwmock simulate config.yaml --dry-run 2>&1 | tail -3

In [ ]:
!gwmock simulate config.yaml --author 'Workshop' --email 'you@example.org' 2>&1 | tail -2

In [ ]:
!find output metadata -type f | sort

Four frame files (signal and noise for each of the two 2L
interferometers) plus a **metadata sidecar** — notebook 05 shows what that
buys you (bit-level reproducibility and validation).

## 3. Look at the data

The frames are standard GWF — read them with GWpy:

In [ ]:
import matplotlib.pyplot as plt
from gwpy.timeseries import TimeSeries

ch = "ET1_2L_ALIGNED_SARD:STRAIN"
sig = TimeSeries.read("output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf", channel=ch)
noi = TimeSeries.read("output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf", channel=ch)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(sig.times.value - sig.t0.value, sig.value, lw=0.5)
axes[0].set_ylabel("signal strain")
axes[1].plot(noi.times.value - noi.t0.value, noi.value, lw=0.3, color="gray")
axes[1].set_ylabel("noise strain")
axes[1].set_xlabel(f"time since GPS {sig.t0.value:.0f} [s]")
axes[0].set_title("ET1 (Sardinia): BBH signal and detector noise")
plt.tight_layout()
plt.show()

The chirp ends at t ≈ 82 s (our `coa_time`). Zoom in on the merger:

In [ ]:
zoom = sig.crop(1577491299, 1577491301)
plt.figure(figsize=(9, 3))
plt.plot(zoom.times.value - 1577491300.0, zoom.value, lw=0.8)
plt.xlabel("time since coalescence [s]")
plt.ylabel("strain")
plt.title("the last two seconds")
plt.show()

## 4. Merge signal + noise into analysis strain

Downstream pipelines want one strain channel. `gwmock merge` adds frames
sample-by-sample:

In [ ]:
!gwmock merge \
    output/signal/E-ET1_2L_ALIGNED_SARD_BBH-1577491218-128.gwf \
    output/noise/E-ET1_2L_ALIGNED_SARD_NOISE-1577491218-128.gwf \
    --channel ET1_2L_ALIGNED_SARD:STRAIN --output strain_E1.gwf --force 2>&1 | tail -1

In [ ]:
import numpy as np

data = TimeSeries.read("strain_E1.gwf", channel=ch)
print("merged == signal + noise :", np.allclose(data.value, sig.value + noi.value))

## 5. Validate outputs against metadata

Every output's hash is recorded in the metadata sidecar — `gwmock validate`
recomputes and compares:

In [ ]:
!gwmock validate output/ --metadata-paths metadata/ 2>&1 | tail -4

## 6. Prefer not to hand-write YAML? — the interactive editor

gwmock also ships a **terminal UI** that builds this same config
interactively, with a live preview and validation
(needs a real terminal — run it locally, not in Colab):

```bash
gwmock config --interactive          # or:  --interactive --load config.yaml
```

Inside the editor, commands start with `/` (Tab completes, ↑/↓ recalls):

| Command | Does |
|---|---|
| `/template noise` (or `signal+noise`, `glitch`) | start from a preset |
| `/psds`, `/geometries` | list available PSDs / detector layouts |
| `/noise psd ET_10_full_cryo_psd` | set the noise curve |
| `/noise detectors ET-Triangle-EMR` | set the network |
| `/save config.yaml` | write the file |
| `/generate-script slurm job.sh` | emit a Slurm launch script |
| `/help` | everything else |

The output is a plain `config.yaml` — exactly what we wrote by hand above.

## ✏️ Try it (5 min)

1. Edit `population.csv`: make it a **100+80 M☉** system at 2000 Mpc.
   Re-run the simulate cell — how does the chirp change?
2. Switch `detectors` to `ET-Triangle-Sardinia` (3 channels: `ET1_SARD`,
   `ET2_SARD`, `ET3_SARD`) and re-run — now you get 6 files.
3. Change `seed` under `noise.arguments` and confirm the noise changes
   while the signal stays identical.

> Tip: delete `output/` and `metadata/` (or use `--force`) when changing
> the config, since a re-run otherwise resumes from the checkpoint.

Next: **02 · Populations** — replace the hand-written CSV with a model.